# 🧠 Implementación de Redes Neuronales Básicas
**Semana 2 – Actividad 1 | Deep Learning**

---

En este cuaderno implementamos desde cero los bloques fundamentales del aprendizaje profundo:

1. **Perceptrón simple** — neurona con función de activación escalón
2. **Red neuronal de una capa** — múltiples neuronas en paralelo
3. **Red neuronal multicapa (MLP)** — capas ocultas con activación sigmoide

Todo el código usa **NumPy** para operaciones matriciales eficientes.

---

## 0. Importaciones

In [ ]:
import numpy as np

---
## 1. Perceptrón Simple

Un **perceptrón** es la unidad mínima de una red neuronal. Recibe entradas $x_i$, las pondera con pesos $w_i$, suma el sesgo $b$ y aplica una función de activación:

$$z = \sum_{i} x_i \cdot w_i + b \qquad y = \text{step}(z) = \begin{cases} 1 & z \geq 0 \\ 0 & z < 0 \end{cases}$$

El perceptrón puede resolver problemas **linealmente separables** como las compuertas lógicas AND y OR.

In [ ]:
# ── Funciones de activación ──────────────────────────────────────────────────

def step(z):
    """Función escalón: devuelve 1 si z >= 0, de lo contrario 0."""
    return np.where(z >= 0, 1, 0)


# ── Perceptrón vectorizado con NumPy ─────────────────────────────────────────

def perceptron(X, W, b):
    """
    Perceptrón simple.

    Parámetros
    ----------
    X : ndarray de forma (n_muestras, n_entradas)
        Matriz de entradas.
    W : ndarray de forma (n_entradas,)
        Vector de pesos.
    b : float
        Sesgo (bias).

    Retorna
    -------
    z : ndarray  — puntaje lineal antes de la activación
    y : ndarray  — salida binaria (0 ó 1)
    """
    z = X @ W + b          # producto matricial: (n_muestras, n_entradas) @ (n_entradas,)
    y = step(z)
    return z, y


# ── Datos: todas las combinaciones binarias de 2 entradas ────────────────────
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]], dtype=float)

etiquetas = ['(0,0)', '(0,1)', '(1,0)', '(1,1)']


# ── Evaluación: compuerta AND  (b = -1.5 es el valor óptimo) ────────────────
print('=== Perceptrón — Compuerta AND ===')
print(f'{"Entrada":>8}  {"z":>6}  {"Salida":>6}  {"AND esperado":>14}')
print('-' * 42)

W_and = np.array([1.0, 1.0])   # pesos iguales: ambas entradas importan igual
b_and = -1.5                   # bias: requiere que z >= 0, es decir x1+x2 >= 1.5
y_and_esperado = np.array([0, 0, 0, 1])   # tabla de verdad AND

z_vals, y_vals = perceptron(X, W_and, b_and)

for i in range(len(X)):
    correcto = '✓' if y_vals[i] == y_and_esperado[i] else '✗'
    print(f'{etiquetas[i]:>8}  {z_vals[i]:>6.1f}  {int(y_vals[i]):>6}  {y_and_esperado[i]:>6}  {correcto}')

exactitud = np.mean(y_vals == y_and_esperado) * 100
print(f'\nExactitud AND: {exactitud:.0f}%')


# ── Evaluación: compuerta OR  (b = -0.5) ────────────────────────────────────
print('\n=== Perceptrón — Compuerta OR ===')
print(f'{"Entrada":>8}  {"z":>6}  {"Salida":>6}  {"OR esperado":>13}')
print('-' * 42)

W_or = np.array([1.0, 1.0])
b_or = -0.5                    # bias más alto: basta con que una entrada sea 1
y_or_esperado = np.array([0, 1, 1, 1])

z_vals_or, y_vals_or = perceptron(X, W_or, b_or)

for i in range(len(X)):
    correcto = '✓' if y_vals_or[i] == y_or_esperado[i] else '✗'
    print(f'{etiquetas[i]:>8}  {z_vals_or[i]:>6.1f}  {int(y_vals_or[i]):>6}  {y_or_esperado[i]:>6}  {correcto}')

exactitud_or = np.mean(y_vals_or == y_or_esperado) * 100
print(f'\nExactitud OR: {exactitud_or:.0f}%')

### 📝 Análisis del Perceptrón

**¿Con qué valor de `b` la neurona se comporta más parecido a AND?**

El valor óptimo es **b = −1.5**. Con este bias, la condición de activación es $x_1 + x_2 \geq 1.5$, que solo se cumple cuando *ambas* entradas valen 1. Si `b > −1`, la neurona se activa con una sola entrada (se parece a OR). Si `b < −2`, nunca se activa.

**¿Qué efecto tiene subir o bajar el bias?**

| Cambio en `b` | Efecto |
|---|---|
| Más positivo (sube) | Umbral de activación más bajo → la neurona se activa con menos entrada |
| Más negativo (baja) | Umbral de activación más alto → la neurona necesita más entrada para disparar |

El bias actúa como el umbral de sensibilidad de la neurona.

---
## 2. Red Neuronal de Una Capa

Una **red de una capa** (también llamada capa densa o *single-layer network*) extiende el perceptrón a varias neuronas en paralelo. Cada neurona aprende una función diferente sobre las mismas entradas.

$$\mathbf{Z} = X \cdot W^T + \mathbf{b} \qquad \mathbf{\hat{Y}} = \sigma(\mathbf{Z})$$

donde $\sigma$ es la función **sigmoide**: $\sigma(z) = \frac{1}{1 + e^{-z}}$, que mapea cualquier valor real a $(0, 1)$.

In [ ]:
# ── Función de activación sigmoide ───────────────────────────────────────────

def sigmoid(z):
    """Sigmoide: mapea z a (0, 1). Útil para salidas de probabilidad."""
    return 1.0 / (1.0 + np.exp(-z))


# ── Capa densa genérica ──────────────────────────────────────────────────────

def capa_densa(X, W, b, activacion=sigmoid):
    """
    Capa totalmente conectada (densa).

    Parámetros
    ----------
    X          : ndarray (n_muestras, n_entradas)
    W          : ndarray (n_neuronas, n_entradas)  — matriz de pesos
    b          : ndarray (n_neuronas,)             — vector de sesgos
    activacion : función                           — función de activación

    Retorna
    -------
    Z : puntaje lineal  (n_muestras, n_neuronas)
    A : activación      (n_muestras, n_neuronas)
    """
    Z = X @ W.T + b        # (n_muestras, n_entradas) @ (n_entradas, n_neuronas)
    A = activacion(Z)
    return Z, A


# ── Red de una sola capa con 3 neuronas ─────────────────────────────────────
# Cada fila de W define los pesos de una neurona; b tiene un valor por neurona.

np.random.seed(42)          # semilla para reproducibilidad
n_entradas  = 2             # x1 y x2
n_neuronas  = 3             # tres neuronas en la capa

W1 = np.random.randn(n_neuronas, n_entradas) * 0.5   # forma (3, 2)
b1 = np.zeros(n_neuronas)                             # forma (3,)

Z1, A1 = capa_densa(X, W1, b1, activacion=sigmoid)

print('=== Red Neuronal de Una Capa ===')
print(f'Entradas X          → forma {X.shape}')
print(f'Pesos W1            → forma {W1.shape}')
print(f'Sesgos b1           → forma {b1.shape}')
print(f'Puntaje Z1          → forma {Z1.shape}')
print(f'Activación A1       → forma {A1.shape}')
print()
print('Pesos W1 (cada fila = una neurona):')
print(np.round(W1, 4))
print()
print('Salidas de la capa (sigmoide) por muestra:')
print(f'{"Entrada":>8}  {"Neurona 1":>10}  {"Neurona 2":>10}  {"Neurona 3":>10}')
print('-' * 46)
for i, etq in enumerate(etiquetas):
    print(f'{etq:>8}  {A1[i,0]:>10.4f}  {A1[i,1]:>10.4f}  {A1[i,2]:>10.4f}')

### 📝 Análisis de la Red de Una Capa

- Cada **fila** de la matriz de pesos $W$ define los pesos de una neurona diferente.
- La multiplicación matricial $X \cdot W^T$ calcula el puntaje $z$ de **todas las neuronas y muestras** de forma simultánea: no se necesita ningún bucle.
- La **sigmoide** produce salidas continuas en $(0, 1)$, interpretables como probabilidades.
- Con pesos aleatorios (sin entrenamiento), las salidas no tienen un significado lógico fijo; el propósito aquí es verificar que las dimensiones y operaciones matriciales sean correctas.

---
## 3. Red Neuronal Multicapa (MLP)

Un **perceptrón multicapa** (MLP) apila varias capas densas. La salida de una capa es la entrada de la siguiente (**propagación hacia adelante** o *forward pass*):

$$\text{Capa 1:} \quad A^{[1]} = \sigma(X \cdot W^{[1]T} + b^{[1]})$$
$$\text{Capa 2:} \quad \hat{y} = \sigma(A^{[1]} \cdot W^{[2]T} + b^{[2]})$$

Las redes multicapa pueden aprender representaciones **no lineales** y resolver problemas que un perceptrón simple no puede (como XOR).

In [ ]:
# ── Red Neuronal Multicapa (MLP) con 2 capas ─────────────────────────────────

class RedNeuronalMLP:
    """
    Perceptrón multicapa de dos capas con función sigmoide.

    Arquitectura: entrada → capa oculta → capa de salida
    """

    def __init__(self, n_entradas, n_ocultas, n_salidas, semilla=42):
        """
        Inicializa los pesos de la red con valores aleatorios pequeños.

        Parámetros
        ----------
        n_entradas : int  — número de características de entrada
        n_ocultas  : int  — neuronas en la capa oculta
        n_salidas  : int  — neuronas en la capa de salida
        """
        rng = np.random.default_rng(semilla)

        # Capa 1 (oculta): pesos (n_ocultas × n_entradas), sesgos (n_ocultas,)
        self.W1 = rng.standard_normal((n_ocultas, n_entradas)) * 0.5
        self.b1 = np.zeros(n_ocultas)

        # Capa 2 (salida): pesos (n_salidas × n_ocultas), sesgos (n_salidas,)
        self.W2 = rng.standard_normal((n_salidas, n_ocultas)) * 0.5
        self.b2 = np.zeros(n_salidas)

    @staticmethod
    def _sigmoid(z):
        """Sigmoide estable numéricamente."""
        return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

    def forward(self, X):
        """
        Propagación hacia adelante (forward pass).

        Parámetros
        ----------
        X : ndarray (n_muestras, n_entradas)

        Retorna
        -------
        A2 : ndarray (n_muestras, n_salidas) — salida de la red
        cache : dict — valores intermedios para inspección
        """
        # Capa oculta
        Z1 = X @ self.W1.T + self.b1          # puntaje capa 1
        A1 = self._sigmoid(Z1)                # activación capa 1

        # Capa de salida
        Z2 = A1 @ self.W2.T + self.b2         # puntaje capa 2
        A2 = self._sigmoid(Z2)                # activación final

        cache = {'Z1': Z1, 'A1': A1, 'Z2': Z2, 'A2': A2}
        return A2, cache

    def predecir(self, X, umbral=0.5):
        """
        Clasificación binaria: aplica un umbral a la salida de la red.

        Parámetros
        ----------
        umbral : float — valor de corte para decidir entre clase 0 y 1
        """
        A2, _ = self.forward(X)
        return (A2 >= umbral).astype(int)

    def resumen(self):
        """Imprime las dimensiones de los parámetros de la red."""
        print('Arquitectura de la red:')
        print(f'  Capa oculta  — W1: {self.W1.shape}, b1: {self.b1.shape}')
        print(f'  Capa salida  — W2: {self.W2.shape}, b2: {self.b2.shape}')
        total = self.W1.size + self.b1.size + self.W2.size + self.b2.size
        print(f'  Total de parámetros: {total}')


# ── Instanciar y evaluar la MLP ──────────────────────────────────────────────
mlp = RedNeuronalMLP(n_entradas=2, n_ocultas=4, n_salidas=1)
mlp.resumen()

print()
A2, cache = mlp.forward(X)
predicciones = mlp.predecir(X)

print('\nResultados del forward pass:')
print(f'{"Entrada":>8}  {"A1 (oculta)":>30}  {"A2 (salida)":>12}  {"Clase":>6}')
print('-' * 68)
for i, etq in enumerate(etiquetas):
    a1_str = '  '.join(f'{v:.3f}' for v in cache['A1'][i])
    print(f'{etq:>8}  [{a1_str}]  {A2[i,0]:>12.4f}  {int(predicciones[i,0]):>6}')

### 📝 Análisis de la Red Multicapa

| Capa | Dimensiones de W | Función |
|---|---|---|
| Oculta (1) | (4 × 2) | Aprende representaciones intermedias |
| Salida (2) | (1 × 4) | Combina representaciones → clasificación final |

- La **capa oculta** transforma el espacio de entrada, permitiendo separar clases que no son linealmente separables.
- Todas las operaciones son **multiplicaciones matriciales con NumPy**: eficientes y vectorizadas.
- Con pesos aleatorios (sin entrenamiento), las salidas son arbitrarias. En un flujo real, se entrenaría con retropropagación (*backpropagation*) para ajustar $W$ y $b$.

---
## 4. Demostración: XOR con MLP

La compuerta **XOR** es un problema clásico **no linealmente separable**: un perceptrón simple no puede resolverlo. Aquí mostramos por qué la red multicapa sí puede (con los pesos correctos).

In [ ]:
# ── XOR con pesos entrenados manualmente ─────────────────────────────────────
# Se usan pesos conocidos que implementan XOR exactamente.

mlp_xor = RedNeuronalMLP(n_entradas=2, n_ocultas=2, n_salidas=1)

# Pesos que implementan XOR:
# Capa oculta: detecta OR (neurona 0) y NAND (neurona 1)
mlp_xor.W1 = np.array([[ 20,  20],    # neurona 0: OR  (activa si x1 OR x2)
                        [-20, -20]])   # neurona 1: NAND (activa si NOT(x1 AND x2))
mlp_xor.b1 = np.array([-10, 30])      # bias correspondientes

# Capa de salida: AND de las dos neuronas anteriores → XOR
mlp_xor.W2 = np.array([[20, 20]])
mlp_xor.b2 = np.array([-30])

y_xor_esperado = np.array([0, 1, 1, 0])   # tabla de verdad XOR

A2_xor, _ = mlp_xor.forward(X)
pred_xor   = mlp_xor.predecir(X)

print('=== MLP resolviendo XOR ===')
print(f'{"Entrada":>8}  {"Salida MLP":>12}  {"XOR esperado":>14}  {"✓/✗":>4}')
print('-' * 44)
for i, etq in enumerate(etiquetas):
    correcto = '✓' if int(pred_xor[i, 0]) == y_xor_esperado[i] else '✗'
    print(f'{etq:>8}  {A2_xor[i,0]:>12.4f}  {y_xor_esperado[i]:>14}  {correcto:>4}')

exactitud_xor = np.mean(pred_xor.flatten() == y_xor_esperado) * 100
print(f'\nExactitud XOR con MLP: {exactitud_xor:.0f}%')
print('\n→ Un perceptrón simple NUNCA puede resolver XOR.')
print('  La capa oculta crea una representación intermedia que lo hace posible.')

---
## 5. Conclusiones

| Modelo | Capacidad | Limitación |
|---|---|---|
| **Perceptrón** | Clasifica problemas linealmente separables (AND, OR) | No puede resolver XOR ni otros no lineales |
| **Red una capa** | Varias neuronas en paralelo sobre las mismas entradas | Sigue siendo lineal si la activación es lineal |
| **MLP (multicapa)** | Aprende representaciones no lineales jerárquicas | Requiere entrenamiento con retropropagación |

**Conceptos clave demostrados:**
- `entradas` → vector $x$ de características
- `pesos` → matriz $W$; cada fila define una neurona
- `sesgo (bias)` → desplaza el umbral de activación
- `puntaje z` → combinación lineal antes de la activación
- `función de activación` → introduce no linealidad (escalón, sigmoide)
- `salida de clasificación` → 0/1 tras aplicar un umbral a la probabilidad

Todas las operaciones se realizaron con **NumPy** usando multiplicaciones matriciales (`@`), sumas vectorizadas y broadcasting, sin bucles explícitos sobre muestras o neuronas.